# ⚡ RQ4: Computational Efficiency & Hardware Profiling
### Hardware-Invariant Complexity ($N_{\text{fwd}}$, $N_{\text{bwd}}$, $\text{FwdU}$) and GPU Profiling (CUDA Events, Peak VRAM)

---
**Research Question 4:** What is the empirical and normalized computational cost of each XAI method, and how do they scale with input sequence length $L$?

Complexity Dimensions:
1. **Hardware-Invariant Metrics**:
   - Forward pass count ($N_{\text{fwd}}$)
   - Backward pass count ($N_{\text{bwd}}$)
   - Asymptotic complexity $\mathcal{O}(\dots)$
2. **Normalized Cost**:
   - $\text{FwdU}$ (multiples of baseline forward latency)
3. **Physical GPU Profiling**:
   - CUDA kernel latency (`torch.cuda.Event`) [ms]
   - Peak GPU memory allocation (`torch.cuda.max_memory_allocated()`) [MB]


## 1. GPU Verification & Baseline Inference Latency Measurement


In [ ]:
!nvidia-smi

import sys
sys.path.insert(0, ".")
from src.models import SentimentCausalLM

model = SentimentCausalLM()
t_base = model.benchmark_baseline_forward(num_trials=20)
print(f"\n---> Measured baseline forward pass latency (T_baseline): {t_base*1000:.2f} ms")


## 2. Profiling All 5 XAI Methods on Benchmark Prompt


In [ ]:
import time
import torch
import pandas as pd
from src.explainers.gradient_x_input import GradientXInputExplainer
from src.explainers.occlusion_explainer import OcclusionExplainer
from src.explainers.integrated_gradients import IntegratedGradientsExplainer
from src.explainers.lime_explainer import LIMEExplainer
from src.explainers.shap_explainer import SHAPExplainer

sample_text = "It is a charming and thoroughly engaging romantic comedy with brilliant acting."

explainers = [
    GradientXInputExplainer(model),
    OcclusionExplainer(model),
    IntegratedGradientsExplainer(model, steps=25),
    LIMEExplainer(model, num_samples=150),
    SHAPExplainer(model, max_evals=1500)
]

records = []
for exp in explainers:
    model.reset_counters()
    torch.cuda.reset_peak_memory_stats()
    
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    start_event.record()
    res = exp.explain(sample_text)
    end_event.record()
    torch.cuda.synchronize()
    
    cuda_time_ms = start_event.elapsed_time(end_event)
    peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
    counters = model.get_counters()
    
    records.append({
        "Method": exp.__class__.__name__.replace("Explainer", ""),
        "CUDA Time (ms)": round(cuda_time_ms, 2),
        "Peak VRAM (MB)": round(peak_vram_mb, 1),
        "Forward Passes": counters["forward_passes"],
        "Backward Passes": counters["backward_passes"],
    })

df_prof = pd.DataFrame(records)
display(df_prof)
